# Preparación de datos para cálculo del indicador Orgullo + Confianza 2025

Este notebook ejecuta los procesos de preparación de datos de la encuesta de indicadores de cultura ciudadana y garantía de derechos, para el cálculo del indicador de Confianza y Orgullo. (O+C).

---
Secretaría de Cultura, Recreación y Deporte
Dirección Observatorio y Gestión del Conocimiento Cultural
Sistemas de Información y Narrativas
Febrero de 2026
observatorio@scrd.gov.co

In [1]:
import pandas as pd

In [ ]:
file_id = 'PUT_YOUR_FILE_ID_HERE'
puntos_gid = '1596209276'
variables_gid = '1958699606'
year = 2025

In [3]:
# URL de googlesheets en formato CSV
url_csv = f'https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv&gid={puntos_gid}'
print(url_csv)
df_datos_puntos_exp = pd.read_csv(url_csv)

https://docs.google.com/spreadsheets/d/1AYWbsOqvwmbAgX-6f1kNckcQ70vT6oz_BmwwpJYW_vE/export?format=csv&gid=1596209276


In [4]:
url_variables = url_csv = f'https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv&gid={variables_gid}'
df_variables = pd.read_csv(url_variables)

In [5]:
# Lista de variable a derretir, es decir, aquellas que tienen un peso asignado
variables_a_derretir = df_variables[df_variables['etiqueta_corta'].notna()]['codigo_variable'].tolist()

In [6]:
df_datos_derretidos = df_datos_puntos_exp.melt(
    id_vars=['encuestado_id', 'localidad', 'V1', 'FACTOR'],
    value_vars=variables_a_derretir,
    var_name='codigo_variable', value_name='sum_factor_success')

# Se agrega la columna 'year'
df_datos_derretidos['year'] = year

# Se agrega la columna etiqueta_corta a partir del dataframe de variables
df_datos_derretidos = df_datos_derretidos.merge(df_variables[['codigo_variable', 'etiqueta_corta']], on='codigo_variable', how='left')

# Se renombra la columna 'etiqueta_corta' a 'key_variable'
df_datos_derretidos.rename(columns={'etiqueta_corta': 'key_variable'}, inplace=True)

# Convertir sum_factor a numérico, reemplazando comas con puntos y manejando errores
df_datos_derretidos['sum_factor_success'] = pd.to_numeric(df_datos_derretidos['sum_factor_success'].str.replace(',', '.'), errors='coerce')
df_datos_derretidos['FACTOR'] = pd.to_numeric(df_datos_derretidos['FACTOR'].str.replace(',', '.'), errors='coerce')

# Cambiar el nombre de la columna 'V1' a 'localidad_cod'
df_datos_derretidos.rename(columns={'V1': 'localidad_cod'}, inplace=True)

In [7]:
# Agrupar por 'localidad', 'key_variable' y 'year', sumando los valores de 'sum_factor_success' y 'FACTOR'
df_agrupado = df_datos_derretidos.groupby(['localidad', 'key_variable', 'year', 'localidad_cod']).agg({'sum_factor_success': 'sum', 'FACTOR': 'sum'}).reset_index()

In [8]:
# Calcular la columna 'porcentaje_success' como el resultado de dividir 'sum_factor_success' entre 'FACTOR'
df_agrupado[f'porcentaje_{year}'] = (df_agrupado['sum_factor_success'] / df_agrupado['FACTOR'])

In [9]:
# Cambiar nombre de la columna sum_factor_success a sum_factor_success_year
df_agrupado.rename(columns={'sum_factor_success': f'sum_factor_success_{year}'}, inplace=True)
df_agrupado.rename(columns={'FACTOR': f'FACTOR_{year}'}, inplace=True)

In [10]:
# Sobre escribir archivo
from openpyxl import load_workbook

# Ruta del archivo Excel
file_path = f'detalle_analisis_indicador_oc_2.xlsx'

# Cargar el archivo Excel existente
book = load_workbook(file_path)

# Utilizamos el engine openpyxl para abrir el archivo sin sobrescribir las otras hojas
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    # Escribimos o actualizamos la hoja de cálculo 'respuestas_detalle'
    df_agrupado.to_excel(writer, index=False, sheet_name=f'datos_{year}')

---
# Consolidación de las tablas 2024 y 2025 Horizontalmente
---

In [11]:
# Ruta del archivo Excel
file_path = f'detalle_analisis_indicador_oc_2.xlsx'

# Agregar la columna porcentaje_2024 al dataframe de datos_2025
df_datos_2025 = pd.read_excel(file_path, sheet_name='datos_2025')
df_datos_2024 = pd.read_excel(file_path, sheet_name='datos_2024')
df_datos_2025 = df_datos_2025.merge(df_datos_2024[['localidad', 'key_variable', 'sum_factor_success_2024', 'FACTOR_2024', 'porcentaje_2024']], on=['localidad', 'key_variable'], how='left')

# Agregar el título de la variable a partir de df_variables
df_datos_2025 = df_datos_2025.merge(df_variables[['codigo_variable', 'etiqueta_corta', 'titulo']], left_on='key_variable', right_on='etiqueta_corta', how='left')
df_datos_2025.drop(columns=['codigo_variable', 'etiqueta_corta'], inplace=True)

In [14]:
df_datos_2025.head()

,localidad,key_variable,year,localidad_cod,sum_factor_success_2025,FACTOR_2025,porcentaje_2025,sum_factor_success_2024,FACTOR_2024,porcentaje_2024,titulo
0,Antonio Nariño,confianza_alcaldia,2025,15,22559.5273,69983.000001,0.322357,9952.32,70062.24,0.142050,Confianza en la alcaldía mayor y entidades
1,Antonio Nariño,confianza_servidores,2025,15,33178.5565,69983.000001,0.474095,14269.42,70062.24,0.203668,Confianza en los servidores del Distrito
2,Antonio Nariño,orgullo_bogota,2025,15,41452.5668,69983.000001,0.592323,55348.79,70062.24,0.789995,Orgullo de vivir en Bogotá
3,Antonio Nariño,orgullo_convivencia,2025,15,35529.1689,69983.000001,0.507683,24000.62,70062.24,0.342561,Orgullo por La convivencia ciudadana
4,Antonio Nariño,orgullo_equipamientos,2025,15,43908.3502,69983.000001,0.627415,39211.67,70062.24,0.559669,Orgullo por Los equipamientos culturales


In [15]:
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_datos_2025.to_excel(writer, index=False, sheet_name=f'detalles_oc')